### Setup & configuration

In [ ]:
import os
import sys
import importlib
import json
import time
import mlflow
import pandas as pd
import torch
import importlib
from IPython.display import display, Markdown, JSON, HTML

# append the parent folder to the system path to allow imports from there
sys.path.append("..")

### INPUT VARIABLES

In [19]:
MODEL_NAME = "../../model/Qwen2.5-1.5B-Instruct"
EMB_MODEL_NAME = "../../model/bge-large-en-v1.5"

# define the absolute path of the LLM model
import os

MODEL_PATH = os.path.abspath(MODEL_NAME)
EMB_MODEL_PATH = os.path.abspath(EMB_MODEL_NAME)

In [ ]:
worklog = """
INC-99102

System monitoring detected intermittent packet loss on core switch cluster in Toronto data node TD-14.

Initial alert triggered at 03:41 AM when ICMP ping failures exceeded threshold (packet loss > 40%) for IP range 10.22.14.0/24.

NOC engineer report:
Multiple devices intermittently responding. Device SW-CORE-19 shows unstable connectivity. SSH access failed twice.

At 03:58 AM, logs show repeated ICMP timeout for 10.22.14.23 and 10.22.14.31.

A technician was dispatched but initial remote diagnostics were inconclusive due to network flapping.

On-site technician report (Alex P.):
Device SW-CORE-19 completely unresponsive to ping and console access.
Physical inspection revealed NIC card failure suspected after overheating warning logs.

Replacement procedure initiated:
- removed faulty network interface card
- installed replacement NIC module
- rebooted device

By 05:10 AM, device restored and stable ping response confirmed.

Customer impact: intermittent service degradation across VLAN 220-245.

Next action: monitor stability for 24 hours and confirm no packet loss recurrence.

"""

# I. HF Models Prompting

### Instantiate the LLM model

In [ ]:
# import os
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # define the absolute path of the LLM model
# MODEL_PATH = os.path.abspath(MODEL_NAME)
# print(f"Loading model {MODEL_PATH}...")

# # instantiate the model and tokenizer with local_files_only=True to ensure it loads from the local path
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH,
#     torch_dtype="auto",
#     device_map="auto",
#     local_files_only=True,
# )

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_PATH,
#     local_files_only=True,
# )

# model.eval()

### Run the experiment

In [7]:
PROMPT_template_str = """
You are a high-fidelity industrial incident reconstruction system.

Extract a complete structured incident representation.

CRITICAL RULES:
- Preserve all telemetry
- Preserve all IPs
- Preserve all device identifiers
- Preserve all technician actions
- Never hallucinate
- Never merge unrelated events
- Each event must be atomic

WORKLOG:
{worklog}

Return structured extraction only.
"""

In [ ]:
from typing import List
from pydantic import BaseModel, Field, ConfigDict

from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


# =========================================================
# SCHEMAS
# =========================================================


class Reasoning(BaseModel):
    # 'extra="forbid"' ensures the model doesn't inject unrequested data
    model_config = ConfigDict(extra="forbid")

    step: str = Field(description="A single step of reasoning")
    confidence: float = Field(description="Confidence score for this step, 0.0 to 1.0")


class IncidentSummary(BaseModel):
    model_config = ConfigDict(extra="forbid")

    summary: str = Field(description="A concise summary of the output")
    reasonings: List[Reasoning] = Field(description="List of reasoning steps leading to the summary")


class Verification(BaseModel):
    fix_required: bool = Field(description="Indicates if a fix is required")
    issues: List[str] = Field(description="List of issues found")


class JudgeSchema(BaseModel):
    faithfulness: float = Field(description="Score for faithfulness, 0.0 to 1.0")
    coverage: float = Field(description="Score for coverage, 0.0 to 1.0")
    coherence: float = Field(description="Score for coherence, 0.0 to 1.0")
    hallucination_detected: bool = Field(description="Indicates if hallucination is detected")
    overall_score: float = Field(description="Overall score, 0.0 to 1.0")
    reasoning: str = Field(description="Reasoning for the evaluation")


# =========================================================
# LLM MODEL
# =========================================================
# 1. Load the model and tokenizer from the local path
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, device_map="auto", local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype="auto", device_map="auto", local_files_only=True)

# 2. Create the pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512, temperature=0.1)

# 3. Integrate with LangChain
llm = HuggingFacePipeline(pipeline=pipe)

In [17]:
import mlflow
import mlflow.transformers
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace


# MLflow experiment setup
mlflow.set_experiment("hf-based-prompting")
mlflow.transformers.autolog()


# Apply structured output
# The framework will attempt to enforce the schema on the local generation
chat_model = ChatHuggingFace(llm=llm)
structured_llm = chat_model.with_structured_output(IncidentSummary)

# start the experiment run
with mlflow.start_run():
    start = time.time()

    response = structured_llm.invoke({"worklog": worklog})

    latency = time.time() - start

    # =====================================================
    # MLflow logging
    # =====================================================

    mlflow.log_param("model_name", os.path.basename(MODEL_NAME))
    mlflow.log_metric("latency_sec", latency)

    mlflow.log_text(worklog, "input.txt")
    mlflow.log_text(response.model_dump_json(indent=2), "response.json")

NotImplementedError: Pydantic schema is not supported for function calling

In [11]:
response

'\nYou are a high-fidelity industrial incident reconstruction system.\n\nExtract a complete structured incident representation.\n\nCRITICAL RULES:\n- Preserve all telemetry\n- Preserve all IPs\n- Preserve all device identifiers\n- Preserve all technician actions\n- Never hallucinate\n- Never merge unrelated events\n- Each event must be atomic\n\nWORKLOG:\n\nINC-99102\n\nSystem monitoring detected intermittent packet loss on core switch cluster in Toronto data node TD-14.\n\nInitial alert triggered at 03:41 AM when ICMP ping failures exceeded threshold (packet loss > 40%) for IP range 10.22.14.0/24.\n\nNOC engineer report:\nMultiple devices intermittently responding. Device SW-CORE-19 shows unstable connectivity. SSH access failed twice.\n\nAt 03:58 AM, logs show repeated ICMP timeout for 10.22.14.23 and 10.22.14.31.\n\nA technician was dispatched but initial remote diagnostics were inconclusive due to network flapping.\n\nOn-site technician report (Alex P.):\nDevice SW-CORE-19 complete

# II. LLM Summary Evaluation

### setup MLflow

In [ ]:
import mlflow

mlflow.set_experiment("hf-based-evaluation")

### Instantiate the LLM model

In [ ]:
import json
import time
import os
import pandas as pd
import torch
import importlib

# load modules/prompts
import llm_evaluation

importlib.reload(llm_evaluation)
from llm_evaluation import HybridLLMEvaluator

MODEL_NAME = "../../model/Qwen2.5-1.5B-Instruct"
EMB_MODEL_NAME = "../../model/bge-large-en-v1.5"
# define the absolute path of the LLM model
MODEL_PATH = os.path.abspath(MODEL_NAME)
EMB_MODEL_PATH = os.path.abspath(EMB_MODEL_NAME)


# define the absolute path of the LLM model
print(
    f"Loading HybridLLMEvaluator ... \
        \n- MODEL_NAME={MODEL_PATH}\n- EMB_MODEL_NAME={EMB_MODEL_PATH}"
)

evaluator = HybridLLMEvaluator(
    embedding_model_name=EMB_MODEL_PATH,
    judge_model_name=MODEL_PATH,
)

### Run the experiment

In [ ]:
import json
import logging
import os
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, ValidationError
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# load modules/prompts
import llm_evaluation

importlib.reload(llm_evaluation)
from llm_evaluation import HybridLLMEvaluator


# RUN EXPERIMENT
start_time = time.time()

# =========================
# MLflow RUN
# =========================

with mlflow.start_run():
    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_param("system", "clean_two_model_pipeline")

    SOURCE_TEXT = """
    Apple released a new AI-enabled MacBook Pro
    featuring improved battery life and a faster M4 chip.
    """

    EXTRACTION = {
        "summary": ("Apple released a new MacBook Pro with an M4 chip and better battery life."),
        "entities": [
            "Apple",
            "MacBook Pro",
            "M4 chip",
        ],
        "key_points": [
            "AI-enabled laptop",
            "Improved battery",
            "Faster processor",
        ],
    }

    EXTRACTION_RUNS = [
        json.dumps(EXTRACTION),
        json.dumps(EXTRACTION),
    ]

    # evaluator = HybridLLMEvaluator(embedding_model_name = EMB_MODEL_PATH, judge_model_name = MODEL_PATH,)

    result = evaluator.evaluate(
        sample_id="sample_001",
        source_text=SOURCE_TEXT,
        extraction=EXTRACTION,
        extraction_runs=EXTRACTION_RUNS,
    )

    print("\nEvaluation Result:")
    print(json.dumps(asdict(result), indent=2))


total_latency = time.time() - start_time

print("\n===== ELAPSED TIME =====\n")
print(f"{total_latency:.2f} seconds")

# Syntax

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, device_map="auto", local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype="auto", device_map="auto", local_files_only=True)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512, temperature=0.1, return_full_text=False)

In [5]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, device_map="auto", local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype="auto", device_map="auto", local_files_only=True)

# 2. Create the pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512, temperature=0.1)

# 3. Integrate with LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# Now you can use the llm as usual
response = llm.invoke("What is LangChain?")
print(response)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 584.64it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force clea

What is LangChain? How does it work?

LangChain is a framework for building large language models that can interact with each other. It allows developers to build complex systems by connecting multiple LLMs together in different ways, creating new applications and services.

It works by providing an API for integrating multiple LLMs into one system. This API allows users to specify the desired behavior of the system based on the input they provide. The API then routes the request through the appropriate LLM(s) based on their capabilities and preferences.
For example, if you wanted to create a chatbot that could answer questions about a specific topic, you would use LangChain to connect several LLMs that specialize in answering questions about that topic. You would also set up the system so that the chatbot only uses responses from those LLMs when generating its own answers.
This approach offers many advantages over traditional approaches to building language models. For instance, it al

In [22]:
from typing import List, Optional

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
)

import torch
import outlines

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =========================================================
# SCHEMAS
# =========================================================


class ReasoningStep(BaseModel):
    model_config = ConfigDict(extra="forbid")

    step_id: int = Field(description="Sequential reasoning step ID")

    step_type: str = Field(description=("Reasoning category such as observation, inference, diagnosis, action, validation, conclusion"))

    title: str = Field(description="Short reasoning title")

    explanation: str = Field(description="Detailed explanation")

    evidence: List[str] = Field(default_factory=list, description="Supporting evidence")

    extracted_entities: List[str] = Field(default_factory=list, description="Entities extracted")

    causal_dependency: Optional[str] = Field(default=None, description="Dependency on prior event")

    temporal_reference: Optional[str] = Field(default=None, description="Timeline reference")

    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score")

    validation_status: str = Field(description="Validation state")


class IncidentSummary(BaseModel):
    model_config = ConfigDict(extra="forbid")

    summary: str = Field(description="Concise structured summary")

    root_cause: Optional[str] = Field(default=None, description="Primary root cause")

    impacted_systems: List[str] = Field(default_factory=list, description="Impacted systems")

    severity: Optional[str] = Field(default=None, description="Incident severity")

    reasonings: List[ReasoningStep] = Field(default_factory=list, description="Structured reasoning trace")


# =========================================================
# LOAD HF MODEL
# =========================================================

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_PATH,
#     local_files_only=True,
# )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     local_files_only=True,
# )


# =========================================================
# OUTLINES MODEL WRAPPER
# =========================================================

outlines_model = outlines.from_transformers(
    model,
    tokenizer,
)


# =========================================================
# STRUCTURED JSON GENERATOR
# =========================================================

generator = outlines.generate.json(
    outlines_model,
    IncidentSummary,
)


# =========================================================
# PROMPT
# =========================================================

PROMPT = """
You are an industrial incident reconstruction system.

TASK:
Generate a structured incident reconstruction report.

RULES:
- Preserve chronology
- Preserve telemetry
- Preserve identifiers
- Preserve technician actions
- Preserve evidence
- Preserve causal relationships
- Do NOT hallucinate

WORKLOG:

INC-88421

Packet loss spike in Montreal cluster EAGG-MTL-02.
Latency 12ms → 241ms.
BGP flaps detected.

RT-EAGG-19:
TEMP=97C
FAN_RPM=0
CRC_ERR=11842

FAN-TRAY-2 failure confirmed.

Technician Maria L replaced fan tray.

BGP reset 10.9.0.1
Convergence 96 sec

Recovery after hardware replacement.
"""


# =========================================================
# GENERATE STRUCTURED OUTPUT
# =========================================================

result = generator(
    PROMPT,
    max_tokens=1024,
)


# =========================================================
# OUTPUT
# =========================================================

print("\n" + "=" * 80)
print("STRUCTURED OUTPUT")
print("=" * 80)

print(result.model_dump_json(indent=2))

AttributeError: module 'outlines' has no attribute 'generate'

In [23]:
import outlines
from outlines import generate
from outlines import models

outlines_model = outlines.from_transformers(model, tokenizer)

generator = generate.json(
    outlines_model,
    IncidentSummary,
)

ImportError: cannot import name 'generate' from 'outlines' (/media/zirox2025/DATA/running/GitHub/Knowledge-base-Extraction/.venv/lib/python3.12/site-packages/outlines/__init__.py)

In [ ]:
# Extract JSON from markdown code blocks if present
if "```json" in extracted:
    result = "```json" + extracted.split("```json")[1].strip()
elif "```" in extracted:
    result = "```" + extracted.split("```")[1].strip()
result